# 🛰️ AeroDamage — Building Damage Assessment
### ResNet50 Transfer Learning · xBD Dataset · 4 Damage Classes

**Run every cell from top to bottom. Do not skip any cell.**

| Cell | What it does |
|------|--------------|
| 1 | Mount Google Drive |
| 2 | Check GPU + install deps |
| 3 | Preprocess xBD → building chips |
| 4 | Train ResNet50 (20 epochs) |
| 5 | Evaluate on test set |
| 6 | Download model + history |
| 7 | Run inference on a sample image |

---
## Cell 1 — Mount Google Drive
This connects Colab to your Drive so we can read your dataset and save the model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── EDIT THIS if your folder is named differently on Drive ──────────────────
BASE_DIR   = '/content/drive/MyDrive/building_damage'
# ───────────────────────────────────────────────────────────────────────────

RAW_DIR    = f'{BASE_DIR}/data/raw'
PROC_DIR   = f'{BASE_DIR}/data/processed'
MODEL_DIR  = f'{BASE_DIR}/models'

os.makedirs(PROC_DIR,  exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Quick sanity check — make sure your dataset is visible
print('\n📂 Checking your dataset folder...')
for split in ['train', 'test']:
    img_dir = os.path.join(RAW_DIR, split, 'images')
    lbl_dir = os.path.join(RAW_DIR, split, 'labels')
    imgs = len(os.listdir(img_dir)) if os.path.isdir(img_dir) else 0
    lbls = len(os.listdir(lbl_dir)) if os.path.isdir(lbl_dir) else 0
    status = '✅' if imgs > 0 and lbls > 0 else '❌ NOT FOUND'
    print(f'  {split}: {imgs} images, {lbls} label files  {status}')

print('\n✅ Drive mounted and paths set!')
print(f'   BASE_DIR  = {BASE_DIR}')
print(f'   RAW_DIR   = {RAW_DIR}')
print(f'   PROC_DIR  = {PROC_DIR}')
print(f'   MODEL_DIR = {MODEL_DIR}')

---
## Cell 2 — Check GPU & Install Dependencies

In [ ]:
import torch

# Check GPU
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU found: {gpu}  ({mem:.1f} GB VRAM)')
else:
    print('⚠️  No GPU detected!')
    print('   Go to Runtime → Change runtime type → T4 GPU → Save')
    raise SystemExit('Please enable GPU before continuing.')

# Install any missing packages (torchvision, sklearn, PIL already in Colab)
!pip install -q fastapi uvicorn python-multipart

import torchvision, sklearn, PIL
print(f'\n📦 Package versions:')
print(f'   torch={torch.__version__}')
print(f'   torchvision={torchvision.__version__}')
print(f'   sklearn={sklearn.__version__}')
print('\n✅ All dependencies ready!')

---
## Cell 3 — Preprocess: Extract Building Chips from xBD

The xBD dataset gives us **full satellite scene images** + **JSON files** with polygon coordinates for every building and its damage label.

This cell:
1. Reads every JSON label file
2. Finds the matching satellite image
3. Crops each building polygon into a small image chip
4. Saves it into the right class folder (`no-damage/`, `minor-damage/`, etc.)

**This takes 15–30 minutes** — run it once, results are saved to Drive.

In [ ]:
import os, json, random, shutil, re
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm

# ── Config ──────────────────────────────────────────────────────────────────
VAL_SPLIT   = 0.15    # 15% of train → validation
PADDING     = 10      # px around each building bounding box
MIN_SIZE    = 10      # skip chips smaller than this
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

CLASS_DIRS = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
XBD_LABEL_MAP = {
    'no-damage':     'no-damage',
    'minor-damage':  'minor-damage',
    'major-damage':  'major-damage',
    'destroyed':     'destroyed',
    'un-classified': None,
}

def get_bbox(coords):
    xs, ys = [p[0] for p in coords], [p[1] for p in coords]
    return min(xs), min(ys), max(xs), max(ys)

def crop_building(img, coords):
    x0, y0, x1, y1 = get_bbox(coords)
    x0 = max(0, int(x0) - PADDING)
    y0 = max(0, int(y0) - PADDING)
    x1 = min(img.width,  int(x1) + PADDING)
    y1 = min(img.height, int(y1) + PADDING)
    if (x1 - x0) < MIN_SIZE or (y1 - y0) < MIN_SIZE:
        return None
    return img.crop((x0, y0, x1, y1))

def parse_coords(feature):
    """Extract polygon coordinates from xBD feature dict."""
    geom = feature.get('geometry', {})
    if geom and geom.get('type') == 'Polygon':
        return geom['coordinates'][0]
    wkt = feature.get('wkt', '')
    if wkt:
        m = re.search(r'POLYGON\s*\(\((.*?)\)\)', wkt)
        if m:
            return [[float(v) for v in p.strip().split()] for p in m.group(1).split(',')]
    return None

def process_split(raw_split, out_split):
    images_dir = os.path.join(raw_split, 'images')
    labels_dir = os.path.join(raw_split, 'labels')
    for cls in CLASS_DIRS:
        os.makedirs(os.path.join(out_split, cls), exist_ok=True)

    label_files = sorted([f for f in os.listdir(labels_dir) if f.endswith('.json')])
    saved, skipped = 0, 0
    all_chips = []

    for lf in tqdm(label_files, desc=os.path.basename(raw_split)):
        stem = lf.replace('.json', '')
        img_path = None
        for ext in ['.png', '.tif', '.tiff', '.jpg']:
            c = os.path.join(images_dir, stem + ext)
            if os.path.isfile(c):
                img_path = c
                break
        if not img_path:
            continue

        try:
            scene = Image.open(img_path).convert('RGB')
        except:
            continue

        with open(os.path.join(labels_dir, lf)) as f:
            data = json.load(f)

        features = data.get('features', {}).get('xy', [])
        for i, feat in enumerate(features):
            props  = feat.get('properties', {})
            damage = props.get('subtype', 'un-classified')
            cls    = XBD_LABEL_MAP.get(damage)
            if cls is None:
                skipped += 1
                continue
            coords = parse_coords(feat)
            if not coords:
                skipped += 1
                continue
            chip = crop_building(scene, coords)
            if chip is None:
                skipped += 1
                continue
            fname = f'{stem}_bld{i:04d}.png'
            out   = os.path.join(out_split, cls, fname)
            chip.save(out, 'PNG')
            all_chips.append((out, cls))
            saved += 1

    print(f'  → Saved {saved:,} chips  ({skipped:,} skipped)')
    return all_chips

def split_val(train_chips):
    """Move 15% of training chips into val/ per class."""
    val_dir = os.path.join(PROC_DIR, 'val')
    for cls in CLASS_DIRS:
        os.makedirs(os.path.join(val_dir, cls), exist_ok=True)
    by_cls = {cls: [] for cls in CLASS_DIRS}
    for path, cls in train_chips:
        by_cls[cls].append(path)
    moved = 0
    for cls, paths in by_cls.items():
        random.shuffle(paths)
        n = max(1, int(len(paths) * VAL_SPLIT))
        for p in paths[:n]:
            dst = os.path.join(val_dir, cls, os.path.basename(p))
            shutil.move(p, dst)
            moved += 1
    print(f'  → Moved {moved:,} chips to val/')

# ── Run ─────────────────────────────────────────────────────────────────────
# Skip if already done (re-running wastes 30 min)
already_done = os.path.isdir(os.path.join(PROC_DIR, 'train', 'no-damage')) and \
               len(os.listdir(os.path.join(PROC_DIR, 'train', 'no-damage'))) > 100

if already_done:
    print('⏩ Preprocessing already done — skipping.')
else:
    print('\n[1/3] Processing TRAIN split...')
    train_chips = process_split(
        os.path.join(RAW_DIR, 'train'),
        os.path.join(PROC_DIR, 'train'))

    print('\n[2/3] Carving out VAL split (15% of train)...')
    split_val(train_chips)

    print('\n[3/3] Processing TEST split...')
    process_split(
        os.path.join(RAW_DIR, 'test'),
        os.path.join(PROC_DIR, 'test'))

# Print final summary
print('\n📊 Dataset Summary:')
total_all = 0
for split in ['train', 'val', 'test']:
    print(f'  {split.upper()}:')
    for cls in CLASS_DIRS:
        folder = os.path.join(PROC_DIR, split, cls)
        n = len(os.listdir(folder)) if os.path.isdir(folder) else 0
        total_all += n
        bar = '█' * min(30, n // 1000)
        print(f'    {cls:20s}: {n:7,}  {bar}')
print(f'\n  TOTAL chips: {total_all:,}')
print('\n✅ Preprocessing complete!')

---
## Cell 4 — Train ResNet50 (Transfer Learning, 20 Epochs)

### What transfer learning means here:
- We load **ResNet50 pre-trained on ImageNet** (it already knows edges, textures, shapes)
- We **freeze all layers** → only our new 4-class head trains (epochs 1–5)
- At epoch 5, we **unfreeze `layer4`** (the deepest feature extractor) for fine-tuning
- This is much faster and more accurate than training from scratch

### How class imbalance is handled:
- `WeightedRandomSampler` → rare classes (destroyed) are sampled more often
- Class-weighted cross-entropy loss → wrong predictions on rare classes cost more

**Expected time: ~2 hours on T4 GPU**

In [ ]:
import os, json, copy, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image
from tqdm.notebook import tqdm

# ── Config ──────────────────────────────────────────────────────────────────
NUM_CLASSES   = 4
BATCH_SIZE    = 64      # 64 fits comfortably in T4 VRAM (16GB)
NUM_EPOCHS    = 20
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
IMG_SIZE      = 224
UNFREEZE_EPOCH = 5      # unfreeze ResNet layer4 after this epoch
CLASS_DIRS    = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
CLASS_NAMES   = ['no_damage', 'minor_damage', 'major_damage', 'destroyed']

device = torch.device('cuda')
print(f'✅ Training on: {torch.cuda.get_device_name(0)}')

# ── Dataset ──────────────────────────────────────────────────────────────────
class XBDDataset(Dataset):
    def __init__(self, root, split, transform=None):
        self.transform = transform
        self.samples   = []
        split_dir = os.path.join(root, split)
        for label_idx, cls in enumerate(CLASS_DIRS):
            cls_dir = os.path.join(split_dir, cls)
            if not os.path.isdir(cls_dir): continue
            for f in os.listdir(cls_dir):
                if f.lower().endswith(('.png','.jpg','.jpeg')):
                    self.samples.append((os.path.join(cls_dir, f), label_idx))
        print(f'  {split:5s}: {len(self.samples):>8,} images')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

# ── Transforms ───────────────────────────────────────────────────────────────
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── Load datasets ─────────────────────────────────────────────────────────────
print('\n📂 Loading datasets...')
train_ds = XBDDataset(PROC_DIR, 'train', train_tf)
val_ds   = XBDDataset(PROC_DIR, 'val',   val_tf)
test_ds  = XBDDataset(PROC_DIR, 'test',  val_tf)

# ── Weighted sampler (fix class imbalance) ───────────────────────────────────
labels       = [train_ds[i][1] for i in range(len(train_ds))]
class_counts = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
class_wts    = 1.0 / (class_counts + 1e-6)
sample_wts   = [class_wts[l] for l in labels]
sampler      = WeightedRandomSampler(sample_wts, len(sample_wts), replacement=True)

print(f'\n📊 Class distribution in train:')
for i, (cls, cnt) in enumerate(zip(CLASS_NAMES, class_counts)):
    pct = cnt / class_counts.sum() * 100
    print(f'   {cls:20s}: {int(cnt):7,}  ({pct:.1f}%)  weight={class_wts[i]:.4f}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,    num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,    num_workers=4, pin_memory=True)

# ── Model: ResNet50 with custom head ─────────────────────────────────────────
print('\n🧠 Building ResNet50 model...')
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Freeze ALL pretrained layers — only train our new head at first
for p in model.parameters():
    p.requires_grad = False

# Replace final FC layer: 2048 → 512 → 4 classes
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(2048, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, NUM_CLASSES)
)
model = model.to(device)

# ── Loss: class-weighted cross-entropy ────────────────────────────────────────
wt_tensor = torch.tensor(class_wts / class_wts.sum() * NUM_CLASSES, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=wt_tensor)

# ── Optimizer: only the trainable head params ─────────────────────────────────
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                       lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                  factor=0.5, patience=3, verbose=True)

# ── Training loop ─────────────────────────────────────────────────────────────
history      = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_val     = float('inf')
best_weights = copy.deepcopy(model.state_dict())

print(f'\n🚀 Starting training for {NUM_EPOCHS} epochs...\n')
print(f'{"Epoch":>6}  {"Train Loss":>10}  {"Train Acc":>9}  {"Val Loss":>8}  {"Val Acc":>7}  {"Time":>6}')
print('─' * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # ── Unfreeze layer4 after warm-up ─────────────────────────────────────
    if epoch == UNFREEZE_EPOCH:
        for p in model.layer4.parameters():
            p.requires_grad = True
        # Re-create optimizer to include newly unfrozen params
        optimizer = optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=LR * 0.1,   # lower LR for fine-tuning pretrained layers
            weight_decay=WEIGHT_DECAY)
        print(f'  [Epoch {epoch}] Unfroze ResNet50 layer4 for fine-tuning (LR → {LR*0.1:.2e})')

    # ── Train ─────────────────────────────────────────────────────────────
    model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        tr_loss    += loss.item() * imgs.size(0)
        tr_correct += out.argmax(1).eq(lbls).sum().item()
        tr_total   += lbls.size(0)
    tr_loss /= tr_total
    tr_acc   = tr_correct / tr_total

    # ── Validate ──────────────────────────────────────────────────────────
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out  = model(imgs)
            loss = criterion(out, lbls)
            v_loss    += loss.item() * imgs.size(0)
            v_correct += out.argmax(1).eq(lbls).sum().item()
            v_total   += lbls.size(0)
    v_loss /= v_total
    v_acc   = v_correct / v_total

    scheduler.step(v_loss)
    elapsed = time.time() - t0

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(v_acc)

    marker = ' ⭐' if v_loss < best_val else ''
    print(f'{epoch:>6}  {tr_loss:>10.4f}  {tr_acc:>9.4f}  {v_loss:>8.4f}  {v_acc:>7.4f}  {elapsed:>5.0f}s{marker}')

    if v_loss < best_val:
        best_val     = v_loss
        best_weights = copy.deepcopy(model.state_dict())
        torch.save(best_weights, os.path.join(MODEL_DIR, 'best_model.pth'))

print('\n✅ Training complete! Best model saved to Drive.')

---
## Cell 5 — Evaluate on Test Set
Loads the best saved model and computes full evaluation metrics.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0a0e1a'
matplotlib.rcParams['axes.facecolor']   = '#111827'
matplotlib.rcParams['text.color']       = '#e2e8f0'
matplotlib.rcParams['axes.labelcolor']  = '#e2e8f0'
matplotlib.rcParams['xtick.color']      = '#8899bb'
matplotlib.rcParams['ytick.color']      = '#8899bb'

# Load best model
model.load_state_dict(best_weights)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in tqdm(test_loader, desc='Evaluating'):
        imgs = imgs.to(device)
        out  = model(imgs)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(lbls.numpy())

test_acc = sum(p==l for p,l in zip(all_preds, all_labels)) / len(all_labels)
report   = classification_report(all_labels, all_preds, target_names=CLASS_NAMES, output_dict=True)
cm       = confusion_matrix(all_labels, all_preds)

print(f'\n🎯 TEST ACCURACY: {test_acc*100:.2f}%')
print(f'   Macro F1:        {report["macro avg"]["f1-score"]*100:.2f}%')
print(f'   Macro Precision: {report["macro avg"]["precision"]*100:.2f}%')
print(f'   Macro Recall:    {report["macro avg"]["recall"]*100:.2f}%')
print('\n' + classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

# ── Plot 1: Loss & Accuracy curves ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0e1a')
epochs_x = list(range(1, len(history['train_loss']) + 1))

ax = axes[0]
ax.plot(epochs_x, history['train_loss'], color='#3b82f6', label='Train Loss', linewidth=2)
ax.plot(epochs_x, history['val_loss'],   color='#f97316', label='Val Loss',   linewidth=2)
ax.set_title('Training & Validation Loss', color='#e2e8f0', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(facecolor='#1a2235', edgecolor='#2a3a5c')
ax.grid(color='#2a3a5c', alpha=0.5)

ax = axes[1]
ax.plot(epochs_x, [v*100 for v in history['train_acc']], color='#22c55e', label='Train Acc', linewidth=2)
ax.plot(epochs_x, [v*100 for v in history['val_acc']],   color='#a78bfa', label='Val Acc',   linewidth=2)
ax.set_title('Training & Validation Accuracy', color='#e2e8f0', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
ax.legend(facecolor='#1a2235', edgecolor='#2a3a5c')
ax.grid(color='#2a3a5c', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Plot 2: Confusion Matrix ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('#0a0e1a')
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)
labels_display = ['No Damage', 'Minor', 'Major', 'Destroyed']
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(labels_display, color='#8899bb')
ax.set_yticklabels(labels_display, color='#8899bb')
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual',    fontsize=11)
ax.set_title('Confusion Matrix', color='#e2e8f0', fontsize=13, fontweight='bold')
for i in range(4):
    for j in range(4):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else '#8899bb', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Plot 3: Per-class F1 bar chart ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('#0a0e1a')
colors = ['#22c55e','#eab308','#f97316','#ef4444']
f1s    = [report[c]['f1-score']*100 for c in CLASS_NAMES]
bars   = ax.bar(labels_display, f1s, color=colors, edgecolor='#2a3a5c', linewidth=1.5, width=0.5)
for bar, val in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', color='#e2e8f0', fontsize=11, fontweight='bold')
ax.set_ylabel('F1-Score (%)')
ax.set_title('Per-Class F1 Score', color='#e2e8f0', fontsize=13, fontweight='bold')
ax.set_ylim(0, 105)
ax.grid(axis='y', color='#2a3a5c', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'per_class_f1.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Save training history JSON (for the web dashboard) ───────────────────────
results = {
    'history':                 history,
    'test_accuracy':           test_acc,
    'classification_report':   report,
    'confusion_matrix':        cm.tolist(),
    'class_names':             CLASS_NAMES,
}
with open(os.path.join(MODEL_DIR, 'training_history.json'), 'w') as f:
    json.dump(results, f, indent=2)

print('\n✅ All charts saved to Drive!')
print(f'   training_history.json  → needed by the web dashboard')

---
## Cell 6 — Download Model Files to Your Computer
Downloads `best_model.pth` and `training_history.json` directly to your Downloads folder.

In [ ]:
from google.colab import files

print('⬇️  Downloading model files...')
print('   (A save dialog will appear for each file)')
print()

files.download(os.path.join(MODEL_DIR, 'best_model.pth'))
files.download(os.path.join(MODEL_DIR, 'training_history.json'))
files.download(os.path.join(MODEL_DIR, 'training_curves.png'))
files.download(os.path.join(MODEL_DIR, 'confusion_matrix.png'))
files.download(os.path.join(MODEL_DIR, 'per_class_f1.png'))

print('✅ Done! Put best_model.pth and training_history.json')
print('   into the models/ folder of your building_damage project.')

---
## Cell 7 — Quick Inference Test (Optional)
Test the model on a single image right here in Colab — upload any satellite image chip.

In [ ]:
from google.colab import files as colab_files
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print('Upload a satellite image chip to test the model:')
uploaded = colab_files.upload()

if uploaded:
    fname = list(uploaded.keys())[0]
    img   = Image.open(fname).convert('RGB')

    # Run inference
    tensor = val_tf(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs  = F.softmax(logits, dim=1).squeeze().cpu().numpy()

    pred_idx   = int(np.argmax(probs))
    pred_label = ['No Damage','Minor Damage','Major Damage','Destroyed'][pred_idx]
    colors_hex = ['#22c55e','#eab308','#f97316','#ef4444']

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.patch.set_facecolor('#0a0e1a')

    ax1.imshow(img)
    ax1.set_title(f'Prediction: {pred_label}', color=colors_hex[pred_idx],
                  fontsize=13, fontweight='bold')
    ax1.axis('off')

    bars = ax2.barh(['No Damage','Minor Damage','Major Damage','Destroyed'],
                    [p*100 for p in probs],
                    color=colors_hex, edgecolor='#2a3a5c')
    ax2.set_xlabel('Confidence (%)', color='#e2e8f0')
    ax2.set_title('Class Probabilities', color='#e2e8f0', fontweight='bold')
    ax2.set_xlim(0, 105)
    for bar, val in zip(bars, probs):
        ax2.text(val*100 + 1, bar.get_y() + bar.get_height()/2,
                 f'{val*100:.1f}%', va='center', color='#e2e8f0', fontsize=10)
    ax2.grid(axis='x', color='#2a3a5c', alpha=0.5)
    ax2.set_facecolor('#111827')

    plt.tight_layout()
    plt.show()
    print(f'\n🎯 Predicted: {pred_label}  (confidence: {probs[pred_idx]*100:.1f}%)')

---
## ✅ All Done!

### What you have now:
- `best_model.pth` — your trained ResNet50 weights
- `training_history.json` — all metrics for the web dashboard
- 3 evaluation charts (curves, confusion matrix, per-class F1)

### Next step — run the web dashboard locally:
```bash
# 1. Put best_model.pth + training_history.json into building_damage/models/
# 2. Start the API:
uvicorn src.app:app --reload --port 8000
# 3. Open frontend/index.html in Chrome
```

### Dashboard features:
- 🔍 Upload any satellite image → see buildings color-coded by damage level
- 📊 Training curves, confusion matrix, per-class F1 — all interactive